# Saved 3PTB scene: receptor, reference, rank-1 pose and search domain

Owner: [DockingMT #52](https://github.com/uibcdf/dockingmt/issues/52).
Can a human inspect the saved 2026-10-20 result in its original receptor frame,
without new docking or manual molecular file conversion?

This is the agreed first scene, not a new scientific experiment. It uses the
audited dry receptor/benzamidinium hypothesis from the preceding human notebook.
The pose has 13 submitted atoms; the observed reference has nine heavy atoms.
No omitted hydrogen coordinates, alignment, protonation or affinity claim is
inferred from a display. Five original poses remain available as frames, with
rank 1 selected initially. Ranking is explicitly `vina`, ascending, in kcal/mol.

Run with the retained ordinary receiving kernel outside the checkout. Set
`DOCKINGMT_SCENE_DATA_ROOT` to the source-free case/data harness,
`DOCKINGMT_EXPECTED_PREFIX` to the wheel receiver, and
`DOCKINGMT_SCENE_OUTPUT` to a new output directory. The portable HTML is the
saved visual artifact; rerun the display cell for a live Jupyter widget.

In [1]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
import sys
from contextlib import ExitStack
from pathlib import Path
from unittest.mock import patch

import dockingmt as dmt
import molsysmt as msm
import molsysviewer as msv
import numpy as np
import pyunitwizard as puw
from IPython.display import display
from vina import Vina

root = Path(os.environ['DOCKINGMT_SCENE_DATA_ROOT']).resolve()
prefix = Path(os.environ['DOCKINGMT_EXPECTED_PREFIX']).resolve()
output = Path(os.environ['DOCKINGMT_SCENE_OUTPUT']).resolve()
assert not (root / 'dockingmt').exists()
assert not (root / 'molsysviewer_dockingmt').exists()
assert Path(dmt.__file__).resolve().is_relative_to(prefix)
output.mkdir(parents=True, exist_ok=False)
sys.path.insert(0, str(root))  # Existing case recipe/data, no production overlay.
from devtools.qualify_3ptb_admission import admit
from molsysviewer_dockingmt import (
    compute_box_wireframe_coordinate_pairs, ensure_runtime,
    set_active_pose, set_reference_visibility, set_search_domain_visibility,
)

# Admission may validate submitted files; it must never search or score here.
guards = ExitStack()
guards.enter_context(patch.object(Vina, 'dock', side_effect=AssertionError('No new search')))
guards.enter_context(patch.object(Vina, 'score', side_effect=AssertionError('No new scoring')))
guards.enter_context(patch.object(dmt, 'dock', side_effect=AssertionError('No new docking')))
versions = {name: metadata.version(name) for name in
            ['dockingmt', 'molsysmt', 'molsysviewer', 'argdigest', 'pyunitwizard', 'vina']}
assert versions['dockingmt'] == '0.0.0+180.g1decee5'
assert versions['molsysmt'] == '0.23.0+78.gc35252d89'
assert versions['molsysviewer'] == '0.24.0+76.ga61b75bc'
print(json.dumps(versions, indent=2))

{
  "dockingmt": "0.0.0+180.g1decee5",
  "molsysmt": "0.23.0+78.gc35252d89",
  "molsysviewer": "0.24.0+76.ga61b75bc",
  "argdigest": "0.15.0+5.g3d1ec53",
  "pyunitwizard": "0.28.1",
  "vina": "1.2.7"
}


## Restore the original result and explicit preparation

The result archive is authenticated before decoding. Preparation repeats the
existing public case recipe and checks its exact submitted PDBQT hashes against
the original result. It does not assign a new chemical hypothesis or run Vina.
The reference is the original observed heavy-atom ligand.

In [2]:
saved_path = root / 'devguide/validation/data/human_3ptb/native_result_2026-10-20.json.gz'
saved_bytes = saved_path.read_bytes()
saved_sha = hashlib.sha256(saved_bytes).hexdigest()
assert saved_sha == '4b1f51cdeb135fd6785a181f6bb09a0d0f631ec3d9de7c169c88703748924249'
original_bytes = gzip.decompress(saved_bytes)
assert hashlib.sha256(original_bytes).hexdigest() == 'ed0eeb3ade25e54f8a5829729578fcbec9f0d73d6c0ceb03d146b0c3492d7fac'
record = json.loads(original_bytes)
result = dmt.DockingResult.from_dict(record)
source_path = root / 'devguide/validation/data/3ptb_prospective/3PTB.pdb.gz'
source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
assert source_sha == '75b1b8e73b4c197b7dcb217fbcfdd39c451860c0fd5a86cdef587c4dbd9972be'
receptor, partners, reference, boxes, admission = admit(source_path)
partner = partners['rigid']
box = boxes['native']
for role, prepared in [('receptor', receptor), ('partner', partner)]:
    assert hashlib.sha256(prepared.to_pdbqt().encode()).hexdigest() == result.provenance['backend_artifacts'][role]['sha256']
dmt.verify_captured_inputs(result.provenance['backend_artifacts'])
assert result.provenance['ranking_policy'] == {'score_name': 'vina', 'ascending': True}
assert len(result.poses) == 5 and result.top_pose.rank == 1
score = result.top_pose.scores['vina']
print({'poses': len(result.poses), 'initial_rank': 1, 'score_name': 'vina',
       'score': score, 'unit': 'kcal/mol', 'new_searches': 0})

{'poses': 5, 'initial_rank': 1, 'score_name': 'vina', 'score': -6.422, 'unit': 'kcal/mol', 'new_searches': 0}


## Build and verify the scene through public tools

DockingMT checks pose/source correspondence and chooses the static receptor
pairing. MolSysMT owns conversion, copying and composition; MolSysViewer owns
loading and rendering. Provider warnings about fields dropped during composition
remain visible: the scene is not a new complete preparation/mechanics archive.

In [3]:
view = dmt.view(result, receptor=receptor, partner=partner,
                reference=reference, search_domain=box, pose_rank=1)
rec_n, lig_n = receptor.n_atoms, partner.n_atoms
ref_n = msm.get(reference, n_atoms=True)
assert (rec_n, lig_n, ref_n, view.player.n_structures) == (2011, 13, 9, 5)
assert view.regions.get('docking_complex').atom_indices == tuple(range(rec_n + lig_n))
assert view.regions.get('reference_ligand').atom_indices == tuple(range(rec_n + lig_n, rec_n + lig_n + ref_n))

coordinates = puw.get_value(view.whole.get(coordinates=True), to_unit='angstrom')
rec_coordinates = puw.get_value(receptor.coordinates, to_unit='angstrom')
ref_coordinates = puw.get_value(msm.get(reference, coordinates=True), to_unit='angstrom')[0]
assert coordinates.shape == (5, 2033, 3)
for frame, pose in enumerate(result.poses):
    np.testing.assert_allclose(coordinates[frame, :rec_n], rec_coordinates, rtol=0, atol=1e-12)
    np.testing.assert_allclose(coordinates[frame, rec_n:rec_n + lig_n],
                               puw.get_value(pose.coordinates, to_unit='angstrom'), rtol=0, atol=1e-12)
    np.testing.assert_allclose(coordinates[frame, rec_n + lig_n:], ref_coordinates, rtol=0, atol=1e-12)

# Explicit, disjoint scene regions based on the checked loaded atom map.
view.whole.set_representation(preset='empty')
rec_region = view.regions.add(atom_indices=list(range(rec_n)), tag='receptor',
                            representation='cartoon', color=0xA8B4C4)
pose_region = view.regions.add(atom_indices=list(range(rec_n, rec_n + lig_n)),
                             tag='selected_pose', representation='ball-and-stick', color=0x2288EE)
ref_region = view.regions.get('reference_ligand')
ref_region.set_representation('ball-and-stick', color=0xDD4499)

rec_system = receptor.to_molecular_system()
pose_system = result.top_pose.to_molecular_system(partner)
for region, system in [(rec_region, rec_system), (pose_region, pose_system), (ref_region, reference)]:
    for attribute in ['atom_id', 'atom_name', 'atom_type', 'group_id', 'group_name']:
        np.testing.assert_array_equal(region.get(element='atom', **{attribute: True}),
                                      msm.get(system, element='atom', **{attribute: True}))
shape = next(item for item in view.shapes.records()
             if item.get('options', {}).get('tag') == 'dockingmt:search_domain')
expected_edges = puw.get_value(compute_box_wireframe_coordinate_pairs(box), to_unit='angstrom')
assert expected_edges.shape == (12, 2, 3)
np.testing.assert_allclose(shape['options']['coordinate_pairs'], expected_edges, rtol=0, atol=1e-12)
assert result.provenance['backend_box']['size'] == [20.0, 20.0, 20.0]
print({'frames': 5, 'atoms': 2033, 'source_identities_checked': True,
       'coordinates_preserved_at_A': 1e-12, 'box_edges': 12, 'box_size_A': [20, 20, 20]})

{'frames': 5, 'atoms': 2033, 'source_identities_checked': True, 'coordinates_preserved_at_A': 1e-12, 'box_edges': 12, 'box_size_A': [20, 20, 20]}


In [4]:
# Python command/state checks; browser-visible behavior must be checked separately.
set_reference_visibility(view, False)
assert not ref_region.visible
set_reference_visibility(view, True)
assert ref_region.visible
set_search_domain_visibility(view, False)
assert not ensure_runtime(view).domain_visible
set_search_domain_visibility(view, True)
set_active_pose(view, 5)
assert view.player.index == 4
set_active_pose(view, 1)
assert view.player.index == 0 and ensure_runtime(view).active_pose_rank == 1
with puw.context(standard_units=['pm', 'fs', 'K', 'mole', 'dalton', 'e', 'kJ/mol', 'degrees']):
    np.testing.assert_allclose(puw.get_value(view.whole.get(coordinates=True), to_unit='angstrom'),
                               coordinates, rtol=0, atol=1e-12)
np.testing.assert_allclose(puw.get_value(view.whole.get(coordinates=True), to_unit='angstrom'),
                           coordinates, rtol=0, atol=1e-12)
assert result.to_dict() == record
assert hashlib.sha256(saved_path.read_bytes()).hexdigest() == saved_sha
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == source_sha
print('Visibility commands, frame 5 -> 1, nondefault units and input/result immutability checked.')
display(view)

Visibility commands, frame 5 -> 1, nondefault units and input/result immutability checked.


## Export and human visual check

The receptor is grey, the selected pose blue, the reference pink and the search
domain blue. Rank 1 is the lowest `vina` score in the saved population; it is not
a ranking by RMSD or a claim of biological affinity. The HTML title names the
score and units. In a live notebook the public addon also provides navigation.

Open the generated self-contained `scene.html` in a browser, or serve its directory
with `python -m molsysviewer.preview OUTPUT_DIRECTORY`. Check that all four objects
are visible, pose and reference can be compared, the initial pose is rank 1 and
the box stays in the same frame. Rotate/zoom the camera; do not align molecules.
For live reference/box toggles and frame switching, use the notebook's public
operations above. Exported HTML and a live Python widget are distinct contexts;
Python addon callbacks are not claimed to execute in a static page.

The execution session has no enabled browser (`cua` inventory empty; integrated
browser unavailable). **Rendering, clipping, browser controls and C6 acceptance
remain pending**, regardless of the successful Python checks or HTML export.

In [5]:
html_path = output / 'scene.html'
title = f'3PTB | rank 1 by vina ascending | {score:.3f} kcal/mol | reference and search domain'
view.export.html(str(html_path), title=title, include_controls=True, include_popout=False)
assert html_path.is_file() and html_path.stat().st_size > 0
state = view.export_state()
(output / 'scene_state.json').write_text(json.dumps(state, indent=2, allow_nan=False))
origins = {}
for name, module in tuple(sys.modules.items()):
    if name.split('.')[0] in ('dockingmt', 'molsysviewer_dockingmt') and getattr(module, '__file__', None):
        origin = str(Path(module.__file__).resolve())
        assert Path(origin).is_relative_to(prefix), (name, origin)
        origins[name] = origin
receipt = {
    'owner': 'uibcdf/dockingmt#52', 'versions': versions, 'origins': origins,
    'wheel_producer': '1decee504b112cfd92026c52dcbe22449de30f4f',
    'source_sha256': source_sha, 'saved_result_gzip_sha256': saved_sha,
    'saved_result_original_sha256': hashlib.sha256(original_bytes).hexdigest(),
    'new_searches': 0, 'new_scoring': 0, 'pose_population': 5,
    'receptor_atoms': rec_n, 'pose_atoms': lig_n, 'reference_atoms': ref_n,
    'scene_atoms': 2033, 'scene_frames': 5, 'selected_rank': 1,
    'ranking': result.provenance['ranking_policy'], 'score_name': 'vina',
    'score': score, 'score_unit': 'kcal/mol', 'box_size_A': [20, 20, 20],
    'coordinate_tolerance_A': 1e-12, 'identity_fields_checked':
        ['atom_id', 'atom_name', 'atom_type', 'group_id', 'group_name'],
    'alignment': False, 'python_controls_checked': True,
    'nondefault_units_checked': True, 'source_and_result_immutable': True,
    'browser_verified': False, 'C6_accepted': False,
    'browser_limitation': 'No enabled browser; integrated browser unavailable',
    'html': {'file': html_path.name, 'bytes': html_path.stat().st_size,
             'sha256': hashlib.sha256(html_path.read_bytes()).hexdigest()},
}
(output / 'receipt.json').write_text(json.dumps(receipt, indent=2, allow_nan=False))
guards.close()
print(json.dumps({key: receipt[key] for key in
                  ['selected_rank', 'score_name', 'score', 'new_searches', 'browser_verified', 'C6_accepted']}, indent=2))
print('Scene exported:', html_path)

{
  "selected_rank": 1,
  "score_name": "vina",
  "score": -6.422,
  "new_searches": 0,
  "browser_verified": false,
  "C6_accepted": false
}
Scene exported: /tmp/dockingmt-3ptb-scene-20261023/output/scene.html
